# Baseline Models — Phase 1 & Phase 2 Evaluation

Systematic test of all baseline PU Learning algorithms on multiple datasets at 1% and 25% positive labeling rates.

- **Phase 1**: Reliable Negative extraction — evaluated by F1 on the returned RN set
- **Phase 2**: Full classification of all unlabeled nodes — evaluated by F1, Precision, Recall, Accuracy

In [ ]:
import os
from models import RCSVM, CCRNE, PU_LP, LP_PUL, MCLS
from dataset_system import DatasetManager
from models_experiment import evaluate_f1_score, evaluate_precision_score, evaluate_phase2
import torch
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
manager = DatasetManager()

DATASETS = ['cora', 'citeseer', 'mnist', 'twitch', 'pubmed']
PERCENTS = ['1pct', '25pct']
NUM_NEG = {'cora': 200, 'citeseer': 200, 'mnist': 300, 'twitch': 200, 'pubmed': 200}

results = []

## Helper Functions

In [ ]:
def run_phase1(model, graph, num_neg, needs_mapping=False, node_list=None):
    """Run Phase 1: train + negative_inference, return RN list and metrics."""
    model.train()
    rn = model.negative_inference(num_neg)
    rn_list = rn.tolist() if isinstance(rn, torch.Tensor) else list(rn)
    
    # MCLS and LP_PUL return internal indices — map to graph node IDs for evaluation
    if needs_mapping and node_list is not None:
        rn_eval = [node_list[i] for i in rn_list if i < len(node_list)]
    else:
        rn_eval = rn_list
    
    f1 = evaluate_f1_score(graph, rn_eval, target_negatives=num_neg)
    precision = evaluate_precision_score(graph, rn_eval, target_negatives=num_neg)
    
    return rn_list, rn_eval, f1, precision


def run_phase2(model, rn_list, graph, needs_mapping=False, node_list=None):
    """Run Phase 2: classify all unlabeled nodes."""
    predictions = model.classify(rn_list)
    
    if needs_mapping and node_list is not None:
        predictions = {node_list[k]: v for k, v in predictions.items() if k < len(node_list)}
    
    phase2 = evaluate_phase2(graph, predictions)
    return phase2


def test_model(model_name, model, graph, num_neg, needs_mapping=False, node_list=None):
    """Full test: Phase 1 + Phase 2, returns result dict."""
    rn_list, rn_eval, f1_p1, prec_p1 = run_phase1(model, graph, num_neg, needs_mapping, node_list)
    phase2 = run_phase2(model, rn_list, graph, needs_mapping, node_list)
    
    return {
        'phase1_f1': f1_p1,
        'phase1_precision': prec_p1,
        'phase1_num_rn': len(rn_list),
        'phase2_f1': phase2['f1'],
        'phase2_precision': phase2['precision'],
        'phase2_recall': phase2['recall'],
        'phase2_accuracy': phase2['accuracy']
    }

---
## RCSVM

Phase 2: Iterative SVM (Li & Liu, 2003) — expands RN via decision function threshold, re-trains until convergence.

In [ ]:
print('=' * 60)
print('RCSVM')
print('=' * 60)

for dataset in DATASETS:
    for pct in PERCENTS:
        path = f'datasets/{dataset}_full_{pct}.pkl'
        if not os.path.exists(path):
            print(f'  SKIP {dataset}/{pct} — file not found')
            continue
        
        graph = manager.load_graph(path)
        num_neg = NUM_NEG[dataset]
        
        model = RCSVM(graph)
        res = test_model('RCSVM', model, graph, num_neg)
        res.update({'model': 'RCSVM', 'dataset': dataset, 'percent': pct})
        results.append(res)
        
        print(f'  {dataset}/{pct}: P1 F1={res["phase1_f1"]:.4f} | P2 F1={res["phase2_f1"]:.4f} Acc={res["phase2_accuracy"]:.4f}')

---
## CCRNE

Phase 2: Weighted Voting SVM (Liu & Peng, 2014) — trains multiple SVMs iteratively, final classification via accuracy-weighted majority vote.

In [ ]:
print('=' * 60)
print('CCRNE')
print('=' * 60)

for dataset in DATASETS:
    for pct in PERCENTS:
        path = f'datasets/{dataset}_full_{pct}.pkl'
        if not os.path.exists(path):
            print(f'  SKIP {dataset}/{pct} — file not found')
            continue
        
        graph = manager.load_graph(path)
        num_neg = NUM_NEG[dataset]
        
        model = CCRNE(graph)
        res = test_model('CCRNE', model, graph, num_neg)
        res.update({'model': 'CCRNE', 'dataset': dataset, 'percent': pct})
        results.append(res)
        
        print(f'  {dataset}/{pct}: P1 F1={res["phase1_f1"]:.4f} | P2 F1={res["phase2_f1"]:.4f} Acc={res["phase2_accuracy"]:.4f}')

---
## PU_LP

Phase 2: Label Propagation (Ma & Zhang, 2017) — uses P ∪ RP as positive seeds, RN as negative seeds, propagates via sklearn LabelPropagation.

**Note**: Uses sparse Katz computation via `scipy.sparse.linalg.spsolve` — works on full PubMed (19717 nodes) without memory issues.

In [ ]:
print('=' * 60)
print('PU_LP')
print('=' * 60)

for dataset in DATASETS:
    for pct in PERCENTS:
        path = f'datasets/{dataset}_full_{pct}.pkl'
        if not os.path.exists(path):
            print(f'  SKIP {dataset}/{pct} — file not found')
            continue
        
        graph = manager.load_graph(path)
        num_neg = NUM_NEG[dataset]
        
        model = PU_LP(graph)
        res = test_model('PU_LP', model, graph, num_neg)
        res.update({'model': 'PU_LP', 'dataset': dataset, 'percent': pct})
        results.append(res)
        
        print(f'  {dataset}/{pct}: P1 F1={res["phase1_f1"]:.4f} | P2 F1={res["phase2_f1"]:.4f} Acc={res["phase2_accuracy"]:.4f}')

---
## MCLS

Phase 2: Iterative LS-SVM (Chaudhari & Shevade, 2012) — trains SVM on P + RN, iteratively refines by adding predicted negatives.

**Note**: MCLS uses `SimpleData` (contiguous 0-based indices). Results are mapped back to original node IDs.

In [ ]:
print('=' * 60)
print('MCLS')
print('=' * 60)

for dataset in DATASETS:
    for pct in PERCENTS:
        path = f'datasets/{dataset}_full_{pct}.pkl'
        if not os.path.exists(path):
            print(f'  SKIP {dataset}/{pct} — file not found')
            continue
        
        graph = manager.load_graph(path)
        num_neg = NUM_NEG[dataset]
        node_list = sorted(list(graph.nodes()))
        
        mcls_data = manager.get_data_for_mcls(graph)
        model = MCLS(mcls_data, k=7, ratio=0.1)
        
        res = test_model('MCLS', model, graph, num_neg, needs_mapping=True, node_list=node_list)
        res.update({'model': 'MCLS', 'dataset': dataset, 'percent': pct})
        results.append(res)
        
        print(f'  {dataset}/{pct}: P1 F1={res["phase1_f1"]:.4f} | P2 F1={res["phase2_f1"]:.4f} Acc={res["phase2_accuracy"]:.4f}')

---
## LP_PUL

Phase 2: Harmonic function label propagation (Carnevali et al., 2021) — solves `f_u = L_uu^{-1} · W_ul · f_l` on the graph adjacency.

**Note**: LP_PUL uses PyG Data format. Results are mapped back to original node IDs.

In [ ]:
print('=' * 60)
print('LP_PUL')
print('=' * 60)

for dataset in DATASETS:
    for pct in PERCENTS:
        path = f'datasets/{dataset}_full_{pct}.pkl'
        if not os.path.exists(path):
            print(f'  SKIP {dataset}/{pct} — file not found')
            continue
        
        graph = manager.load_graph(path)
        num_neg = NUM_NEG[dataset]
        node_list = sorted(list(graph.nodes()))
        
        lp_pul_data = manager.get_data_for_lp_pul(graph)
        model = LP_PUL(lp_pul_data)
        
        res = test_model('LP_PUL', model, graph, num_neg, needs_mapping=True, node_list=node_list)
        res.update({'model': 'LP_PUL', 'dataset': dataset, 'percent': pct})
        results.append(res)
        
        print(f'  {dataset}/{pct}: P1 F1={res["phase1_f1"]:.4f} | P2 F1={res["phase2_f1"]:.4f} Acc={res["phase2_accuracy"]:.4f}')

---
## Summary Results

In [ ]:
df = pd.DataFrame(results)
cols = ['model', 'dataset', 'percent', 'phase1_f1', 'phase1_precision', 'phase1_num_rn',
        'phase2_f1', 'phase2_precision', 'phase2_recall', 'phase2_accuracy']
df = df[cols]
df

In [ ]:
print('\n=== Phase 1 F1 (pivot) ===')
pivot_p1 = df.pivot_table(index=['dataset', 'percent'], columns='model', values='phase1_f1')
print(pivot_p1.to_string())

print('\n=== Phase 2 F1 (pivot) ===')
pivot_p2 = df.pivot_table(index=['dataset', 'percent'], columns='model', values='phase2_f1')
print(pivot_p2.to_string())

print('\n=== Phase 2 Accuracy (pivot) ===')
pivot_acc = df.pivot_table(index=['dataset', 'percent'], columns='model', values='phase2_accuracy')
print(pivot_acc.to_string())